# MuseGAN: generowanie krótkich, polifonicznych fraz

## Case study: wielościeżkowy pianoroll w stylu Bach Chorales

MuseGAN to rodzina generatywnych sieci adversarialnych zaprojektowanych dla wielościeżkowej, polifonicznej muzyki. W oryginalnym ujęciu generator tworzy kilka zsynchronizowanych partii (np. instrumentów), a krytyk ocenia ich wiarygodność. Ten notebook buduje mały, edukacyjny wariant inspirowany MuseGAN: generator mapuje wektor szumu do pianorollu o osiach `[czas, wysokość]`, a krytyk odróżnia przykłady prawdziwe od syntetycznych.

### Dlaczego GAN dla muzyki?

GAN uczy generatora $G(z)$ dopasowania rozkładu danych bez jawnego likelihood. Krytyk $D(x)$ otrzymuje przykłady rzeczywiste i wygenerowane, a gradient jego oceny prowadzi generator ku obszarowi danych. W standardowym GAN obie sieci grają minimax; w praktyce użyjemy hinge loss oraz spektralnej normalizacji w krytyku dla stabilniejszej demonstracji.

### Polifonia, takt i reprezentacja

Wielogłosowa partytura nie jest jednym strumieniem nut. Partie muszą zachować synchronię rytmu i współbrzmienia. Pianoroll reprezentuje nuty jako binarną macierz czasu i pitch; poszerzenie do `[track, bar, time, pitch]` daje naturalny format wielościeżkowy. Dla prostoty nasz miniaturowy MuseGAN emituje pojedynczy wspólny pianoroll z kilkoma aktywnymi nutami na krok.

### Trening adversarialny i typowe problemy

Krytyk aktualizuje się na realnych i fałszywych partiach; generator aktualizuje się tak, by krytyk przypisywał jego partiom wysoką ocenę. Różnica liczby aktualizacji, learning rate i pojemności sieci może destabilizować grę. Mode collapse oznacza, że wiele wektorów szumu daje podobne wyniki. W małym przykładzie sprawdzimy aktywność nut i różnorodność próbek zamiast obiecywać wysoką jakość muzyczną.

W oryginalnych wariantach MuseGAN stosuje się między innymi generatory wielościeżkowe, latent codes współdzielone między taktami i ścieżkami oraz temporalne moduły generujące ciągłość. Poniższy model jest celowo mniejszy i nie jest wierną reprodukcją architektury z publikacji.

### Dane i ograniczenia demonstracji

Wspólny loader próbuje pobrać JSB Chorales, a przy braku dostępu generuje oznaczony syntetyczny korpus harmonii. Dyskryminator widzi krótkie fragmenty pianoroll. Krótkie uruchomienie na CPU jest wykonalne, ale adversarialna jakość wymaga zwykle większych danych, strojenia, checkpointów i odsłuchowej oceny. Nie traktuj wykresów loss jako samodzielnego rankingu jakości.

## 1. Dane: JSB Chorales i reprezentacja pianoroll

Bach Chorales to klasyczny mały zbiór symboliczych partytur czterogłosowych. Każdy krok czasowy opisuje wysokości nut w głosach sopranu, altu, tenoru i basu. W tym case study pracujemy na syntetycznej, uproszczonej reprezentacji pianoroll: tensor `[czas, wysokość]`, gdzie 1 oznacza aktywną nutę. Loader najpierw próbuje pobrać plik CSV ze zbioru JSB Chorales. Jeśli sieć lub adres źródłowy jest niedostępny, generuje deterministyczne przykładowe sekwencje harmonii; komórka informuje, który wariant został użyty. To pozwala prześledzić cały pipeline bez ukrywania ograniczeń demonstracyjnego fallbacku.

Przy prawdziwym zbiorze zakładamy CSV z komórkami zawierającymi listy nut MIDI aktywnych w kolejnych krokach. Dla stabilności przykład mapuje nuty do 48 klas wysokości (MIDI 48–95), a sekwencje do stałej długości. Jest to uproszczenie: nie kodujemy długości nut, dynamiki, artykulacji ani dokładnego czasu.

**Co pokazuje ta komórka:** importuje zależności, ustawia ziarna losowości i buduje niewielki zbiór przykładowych partii w postaci wielogłosowych pianorolli. Możesz włączyć próbę pobrania JSB przez ustawienie `TRY_DOWNLOAD=True`.


In [ ]:
import math, random, urllib.request, csv, io
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED=23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)

PITCH_MIN, N_PITCH, STEPS = 48, 48, 32
TRY_DOWNLOAD=False  # Ustaw True, gdy chcesz spróbować pobrać publiczne dane.

def make_demo_corpus(n=240):
    # Cztery głosy, diatoniczne progresje i rytmiczne podtrzymania: kontrolowany fallback.
    scales=[0,2,4,5,7,9,11]
    roots=[0,5,7,0,9,5,0,7]
    rows=[]
    for _ in range(n):
        roll=np.zeros((STEPS,N_PITCH),dtype=np.float32)
        trans=random.choice([-2,0,0,0,2])
        for t in range(STEPS):
            beat=t//4; root=roots[beat%len(roots)]
            chord=[root,(root+4)%12,(root+7)%12,(root+11)%12]
            pitches=[60+root+trans, 60+chord[1]+trans, 48+chord[2]+trans, 48+root+trans]
            for voice,p in enumerate(pitches):
                p=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,p))
                # Zmienność oktawy/tonu i okazjonalne pauzy imitują warianty partii.
                q=p + (12 if voice<2 and random.random()<.25 else 0)
                q=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,q))
                if random.random()>.035: roll[t,q-PITCH_MIN]=1
        rows.append(roll)
    return np.stack(rows)

def load_jsb_or_demo():
    if TRY_DOWNLOAD:
        urls=["https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-train.csv"]
        for url in urls:
            try:
                raw=urllib.request.urlopen(url,timeout=8).read().decode("utf-8")
                # Ten public format przechowuje po jednej obserwacji w wierszu.
                parsed=[]; sequence=[]
                for row in csv.reader(io.StringIO(raw)):
                    vals=[int(x) for x in row if x.strip().lstrip('-').isdigit()]
                    if not vals:
                        if len(sequence)>=STEPS:
                            for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                                parsed.append(np.stack(sequence[start:start+STEPS]))
                        sequence=[]; continue
                    frame=np.zeros(N_PITCH,dtype=np.float32)
                    for p in vals:  # kilka głosów/nut w jednym kroku
                        if PITCH_MIN<=p<PITCH_MIN+N_PITCH: frame[p-PITCH_MIN]=1
                    sequence.append(frame)
                if len(sequence)>=STEPS:
                    for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                        parsed.append(np.stack(sequence[start:start+STEPS]))
                if len(parsed)>20: return np.stack(parsed),"pobrane JSB Chorales (CSV; kroki wielogłosowe)"
            except Exception as e: print("Pobieranie nieudane:",type(e).__name__,str(e)[:120])
    return make_demo_corpus(),"fallback demonstracyjny (syntetyczne harmonie)"

rolls,data_source=load_jsb_or_demo()
print("Źródło:",data_source,"| liczba sekwencji:",len(rolls),"| kształt:",rolls.shape)
plt.figure(figsize=(10,2.5)); plt.imshow(rolls[0].T,aspect="auto",origin="lower",cmap="magma"); plt.xlabel("krok"); plt.ylabel("wysokość MIDI (indeks)" ); plt.title("Przykładowy pianoroll"); plt.show()


## 2. Dane jako tensory realnych fraz

Krytyk pracuje na stałej długości pianorollu. Każda obserwacja zawiera 32 kroki i 48 klas wysokości; wartości są binarne. Używamy losowych fragmentów ze zbioru treningowego. Zachowujemy osobne przykłady walidacyjne do podglądu, choć GAN-y nie mają tak prostego validation loss jak klasyfikator.

**Co pokazuje ta komórka:** przygotowuje tensor danych i batch prawdziwych fragmentów. W razie użycia prawdziwych korali sprawdź histogram aktywnych nut oraz dopasuj zakres wysokości i siatkę czasu do źródła.


In [ ]:
perm=np.random.permutation(len(rolls)); cut=max(1,int(.85*len(rolls)))
train_np,val_np=rolls[perm[:cut]],rolls[perm[cut:]]
train=torch.tensor(train_np,device=device)
LATENT=64; BATCH=32

def real_batch():
    ids=torch.randint(0,len(train),(BATCH,),device=device)
    return train[ids]
real=real_batch()
print("źródło:",data_source,"train:",train.shape,"batch:",real.shape,"aktywne nuty/partię:",real.sum((1,2)).float().mean().item())


## 3. Generator i krytyk

Generator otrzymuje losowy wektor `z` i stopniowo mapuje go na macierz pianorollu. Sigmoid daje prawdopodobieństwa aktywności nut. Krytyk zwraca jedną nieograniczoną ocenę na przykład, a nie prawdopodobieństwo po sigmoidzie. Spektralna normalizacja ogranicza skalę operatora warstw liniowych i może poprawiać stabilność.

**Co pokazuje ta komórka:** definiuje niewielki MLP-generator i konwolucyjnego krytyka. Wariant MuseGAN z osobnymi generatorami dla ścieżek i temporalnymi modułami byłby kolejnym krokiem.


In [ ]:
from torch.nn.utils import spectral_norm

class Generator(nn.Module):
    def __init__(self,z_dim=LATENT):
        super().__init__(); self.net=nn.Sequential(
            nn.Linear(z_dim,256),nn.BatchNorm1d(256),nn.LeakyReLU(.2),
            nn.Linear(256,512),nn.BatchNorm1d(512),nn.LeakyReLU(.2),
            nn.Linear(512,STEPS*N_PITCH),nn.Sigmoid())
    def forward(self,z): return self.net(z).view(-1,1,STEPS,N_PITCH)

class Critic(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(
            spectral_norm(nn.Conv2d(1,32,4,2,1)),nn.LeakyReLU(.2),
            spectral_norm(nn.Conv2d(32,64,4,2,1)),nn.LeakyReLU(.2),
            nn.Flatten(),spectral_norm(nn.Linear(64*8*12,1)))
    def forward(self,x): return self.net(x).squeeze(1)

G=Generator().to(device); D=Critic().to(device)
print("G params:",sum(p.numel() for p in G.parameters()),"D params:",sum(p.numel() for p in D.parameters()))
print("kształt wygenerowanej partii:",G(torch.randn(4,LATENT,device=device)).shape)


## 4. Hinge objective i naprzemienne aktualizacje

Dla krytyka hinge loss to $E[\max(0,1-D(x))]+E[\max(0,1+D(G(z)))]$. Generator minimalizuje $-E[D(G(z))]$. `detach()` podczas kroku krytyka zapobiega aktualizacji generatora; podczas kroku generatora parametry krytyka są zamrożone, ale gradient przepływa przez jego obliczenia do wejściu generatora.

**Co pokazuje ta komórka:** wykonuje kilka krótkich epok aktualizacji. Ograniczamy liczbę kroków i rozmiar modelu, aby demonstracja nie trenowała się godzinami. Lossy w GAN nie muszą monotonicznie maleć i nie należy interpretować ich jak supervised accuracy.


In [ ]:
optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(0.0,.9))
optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(0.0,.9))
EPOCHS=12; STEPS_PER_EPOCH=20; d_hist=[]; g_hist=[]
for epoch in range(EPOCHS):
    dl=[]; gl=[]
    for _ in range(STEPS_PER_EPOCH):
        # 1) Krytyk: odsuń real i fake po przeciwnych stronach marginesu.
        for p in D.parameters(): p.requires_grad_(True)
        xr=real_batch().unsqueeze(1); z=torch.randn(BATCH,LATENT,device=device)
        xf=G(z).detach(); dreal=D(xr); dfake=D(xf)
        lossD=F.relu(1-dreal).mean()+F.relu(1+dfake).mean()
        optD.zero_grad(set_to_none=True); lossD.backward(); optD.step()
        # 2) Generator: zwiększ ocenę krytyka dla próbek fake.
        for p in D.parameters(): p.requires_grad_(False)
        z=torch.randn(BATCH,LATENT,device=device); fake=G(z)
        lossG=-D(fake).mean()
        optG.zero_grad(set_to_none=True); lossG.backward(); optG.step()
        dl.append(lossD.item()); gl.append(lossG.item())
    d_hist.append(float(np.mean(dl))); g_hist.append(float(np.mean(gl)))
    print(f"epoka {epoch+1:02d}/{EPOCHS} | D hinge={d_hist[-1]:.3f} | G={g_hist[-1]:.3f}")


## 5. Podgląd próbek, aktywność i różnorodność

Warto oceniać zarówno strukturę pojedynczego pianorollu, jak i różnice między próbkami. Binarny próg może być dobrany do wizualizacji; nie jest uniwersalnym dekoderem MIDI. Zmierzymy średnią liczbę nut i średnią odległość XOR między parami, co daje prosty sygnał o aktywności i różnorodności.

**Co pokazuje ta komórka:** generuje małą siatkę próbek, porównuje je z realnym batch i liczy proste statystyki. Podobne obrazy lub bliska zeru różnorodność mogą wskazywać na zapadanie się trybów, choć te miary nie oceniają harmonii.


In [ ]:
@torch.no_grad()
def sample(n=8,threshold=.5):
    G.eval(); x=G(torch.randn(n,LATENT,device=device)).squeeze(1)
    G.train(); return (x>threshold).float().cpu().numpy()

fake=sample(8); real_show=real[:4].cpu().numpy()
fig,axes=plt.subplots(2,4,figsize=(12,5))
for i,ax in enumerate(axes.flat):
    arr=fake[i] if i<4 else real_show[i-4]
    ax.imshow(arr.T,aspect="auto",origin="lower",cmap="magma"); ax.set_title("fake" if i<4 else "real")
    ax.set_xlabel("czas"); ax.set_ylabel("pitch")
plt.tight_layout(); plt.show()
flat=fake.reshape(len(fake),-1)
pairs=[np.mean(flat[i]!=flat[j]) for i in range(len(flat)) for j in range(i+1,len(flat))]
print("średnia aktywność fake:",round(float(fake.sum((1,2)).mean()),1),"nut/utwór")
print("średnia odległość XOR między próbkami:",round(float(np.mean(pairs)),3))


## 6. Lossy i interpretacja treningu

W GAN-ach krzywe generatora i krytyka są sprzężone; pojedyncza wartość nie oznacza jakości wprost. Jeśli krytyk zbyt szybko dominuje, generator może otrzymywać słaby lub niestabilny sygnał. Jeśli generator produkuje jedną rodzinę wyników, warto rozważyć większą różnorodność danych, zmianę architektury i regularizację.

**Co pokazuje ta komórka:** rysuje przebieg obu funkcji celu i zestawia go z wizualnymi/statystycznymi próbkami. Należy oceniać wyniki wieloma sposobami, włącznie z odsłuchem po konwersji MIDI.


In [ ]:
fig,ax=plt.subplots(figsize=(7,3)); ax.plot(d_hist,label="krytyk hinge"); ax.plot(g_hist,label="generator")
ax.set_xlabel("epoka"); ax.set_ylabel("loss (skale różne)"); ax.grid(); ax.legend(); plt.show()


## Key takeaways

- MuseGAN należy do rodziny modeli adversarialnych; generator tworzy pianorolle z latentnego szumu, a krytyk uczy się odróżniać próbki od danych.
- Hinge loss i spektralna normalizacja są praktycznymi elementami stabilizującymi małą demonstrację, ale nie gwarantują zbieżności.
- Reprezentacja pianoroll zachowuje współwystępowanie nut, lecz gubi wiele szczegółów wykonania MIDI.
- Loss GAN nie jest prostą miarą jakości. Oceniaj aktywność, różnorodność, strukturę muzyczną, przykłady z walidacji i odsłuch.
- Ten wariant jest inspirowany MuseGAN, nie wierną reprodukcją: pełniejszy model używa wielościeżkowej reprezentacji, kodów współdzielonych oraz modułów temporalnych.
- Dalsza praca: parser MIDI i prawdziwy podział JSB, oddzielne ścieżki głosów, kontrola tonacji/tempa, regularizacja gradientu, checkpointing oraz eksport do MIDI i odsłuch wygenerowanych fragmentów.